# So sánh Classical vs Quantum K-Means

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in [
    "numpy", "pandas", "scikit-learn", "matplotlib", "seaborn",
    "qiskit==1.4.2",
    "qiskit-aer-gpu==0.15.1",
    "qiskit-machine-learning==0.7.2",
]:
    try:
        install(pkg)
        print(f"{pkg} installed success !")
    except Exception as e:
        print(f"{pkg} — {e}")

print("\nFinish.")

In [ ]:
# ==============================================================================
# CELL 1: Import + Setup
# ==============================================================================
import warnings, pickle, os
import numpy as npm
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score
)
warnings.filterwarnings('ignore')

# Màu sắc
COLOR_CL  = '#1a1a2e'
COLOR_QK  = '#e94560'
COLOR_ARI = '#0f3460'
BG        = '#f8f9fa'
PALETTE   = ['#e94560','#0f3460','#533483','#e8c547',
             '#16213e','#a8dadc','#457b9d','#f4a261','#2a9d8f']

print('Setup OK')

In [ ]:
# ==============================================================================
# CELL 2: Mount Drive + Load data
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

ARTIFACT_PATH  = '/content/drive/MyDrive/...'   # path đến artifacts
CHECKPOINT_DIR = '/content/drive/MyDrive/...'   # path đến qkm_checkpoints

with open(ARTIFACT_PATH, 'rb') as f:
    artifacts = pickle.load(f)

pca_scores = artifacts['pca_scores']
pca_cutoff = artifacts['pca_cutoff']
X_full     = pca_scores.values.astype(float)
X          = X_full[:, :pca_cutoff]

K_RANGE = range(2, 11)
ks      = list(K_RANGE)

print(f'X shape    : {X.shape}')
print(f'pca_cutoff : {pca_cutoff}')

In [ ]:
# ==============================================================================
# CELL 2.5: Định nghĩa class để pickle load được
# ==============================================================================
from sklearn.cluster import KMeans as _SKLearnKMeans

class ClassicalKMeansWithHistory:
    def __init__(self, n_clusters, n_init=30, max_iter=300, random_state=123):
        self.n_clusters, self.n_init   = n_clusters, n_init
        self.max_iter, self.random_state = max_iter, random_state
        self.labels_ = self.cluster_centers_ = self.inertia_ = None
        self.best_run_history_ = []
        self.convergence_history_ = []

    def fit(self, X):
        pass  # Không cần fit, chỉ cần class tồn tại để pickle load

class QiskitSwapTestKMeans:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
        self.labels_              = None
        self.cluster_centers_     = None
        self.inertia_             = None
        self.best_run_history_    = []
        self.convergence_history_ = []

print('Classes defined OK')

In [ ]:
# ==============================================================================
# CELL 3: Load tất cả checkpoint
# ==============================================================================
def load_ckpt(name):
    path = f'{CHECKPOINT_DIR}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None

classical_results = {}
quantum_results   = {}

print('Load Classical checkpoints:')
for k in K_RANGE:
    ckpt = load_ckpt(f'classical_k{k}')
    if ckpt is not None:
        classical_results[k] = ckpt
        print(f'  k={k:2d} | inertia={ckpt.inertia_:.2f} | iters={len(ckpt.best_run_history_)}')
    else:
        print(f'  k={k:2d} | MISSING!')

print('\nLoad Quantum checkpoints:')
for k in K_RANGE:
    ckpt = load_ckpt(f'quantum_k{k}')
    if ckpt is not None:
        quantum_results[k] = ckpt
        print(f'  k={k:2d} | inertia={ckpt.inertia_:.4f} | iters={len(ckpt.best_run_history_)}')
    else:
        print(f'  k={k:2d} | MISSING!')

# Chỉ giữ k có đủ cả 2
valid_ks = [k for k in ks if k in classical_results and k in quantum_results]
print(f'\nValid k: {valid_ks}')

In [ ]:
# ==============================================================================
# CELL 4: Tính metrics
# ==============================================================================
records = []
for k in valid_ks:
    cl, qkm = classical_results[k], quantum_results[k]
    ari = adjusted_rand_score(cl.labels_, qkm.labels_)
    for method, m in [('Classical', cl), ('Quantum', qkm)]:
        records.append(dict(
            k=k, method=method,
            inertia           = m.inertia_,
            silhouette        = silhouette_score(X, m.labels_),
            davies_bouldin    = davies_bouldin_score(X, m.labels_),
            calinski_harabasz = calinski_harabasz_score(X, m.labels_),
            ari               = ari,
        ))

metrics_df = pd.DataFrame(records)
cl_df  = metrics_df[metrics_df.method=='Classical'].set_index('k')
qk_df  = metrics_df[metrics_df.method=='Quantum'].set_index('k')
ari_arr = [adjusted_rand_score(classical_results[k].labels_, quantum_results[k].labels_) for k in valid_ks]

print(metrics_df.round(4).to_string(index=False))

In [ ]:
# ==============================================================================
# CELL 5: Fig 1 — So sánh tổng thể
# ==============================================================================
import numpy as np

def norm01(arr):
    a = np.array(arr, dtype=float)
    r = a.max() - a.min()
    return (a - a.min()) / r if r > 1e-10 else np.ones_like(a)

def style_ax(ax, title, ylabel, note):
    ax.set_facecolor('#fff')
    ax.set_title(f'{title}\n{note}', fontsize=12, fontweight='bold', color='#1a1a2e', pad=8)
    ax.set_xlabel('k', fontsize=10); ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(valid_ks); ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', ls='--', alpha=0.4, color='#ccc'); ax.set_axisbelow(True)

def plot_metric(ax, col, title, ylabel, better='high'):
    cv, qv = cl_df[col].values, qk_df[col].values
    fn = np.argmax if better=='high' else np.argmin
    note = 'cao hơn tốt hơn' if better=='high' else 'thấp hơn tốt hơn'
    ax.plot(valid_ks, cv, 'o-',  color=COLOR_CL, lw=2, ms=7, label='Classical', zorder=3)
    ax.plot(valid_ks, qv, 's--', color=COLOR_QK, lw=2, ms=7, label='Quantum',   zorder=3)
    for vals, c_ in [(cv,COLOR_CL),(qv,COLOR_QK)]:
        bk=valid_ks[fn(vals)]; bv=vals[fn(vals)]
        ax.axvline(bk,color=c_,lw=1,ls=':',alpha=0.5)
        ax.annotate(f'k={bk}',xy=(bk,bv),xytext=(4,6),textcoords='offset points',
                    fontsize=8,color=c_,fontweight='bold')
    style_ax(ax, title, ylabel, note); ax.legend(fontsize=9)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.patch.set_facecolor(BG)
plot_metric(axes[0,0], 'silhouette',        'Silhouette Score',        'Score', 'high')
plot_metric(axes[0,1], 'davies_bouldin',    'Davies-Bouldin Index',    'Index', 'low')
plot_metric(axes[1,0], 'calinski_harabasz', 'Calinski-Harabasz Index', 'Index', 'high')

ax4=axes[1,1]; ax4b=ax4.twinx()
ax4.plot(valid_ks,cl_df['inertia'].values,'o-', color=COLOR_CL,lw=2,ms=7,label='Classical inertia')
ax4.plot(valid_ks,qk_df['inertia'].values,'s--',color=COLOR_QK,lw=2,ms=7,label='Quantum inertia')
bars=ax4b.bar(valid_ks,ari_arr,alpha=0.25,color=COLOR_ARI,width=0.5,label='ARI')
for bar,v in zip(bars,ari_arr):
    ax4b.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.02,f'{v:.2f}',
              ha='center',fontsize=8,color=COLOR_ARI)
ax4b.set_ylabel('ARI',color=COLOR_ARI,fontsize=10); ax4b.set_ylim(0,1.5)
ax4b.tick_params(axis='y',labelcolor=COLOR_ARI)
style_ax(ax4,'Elbow + ARI','Inertia','')
l1,lb1=ax4.get_legend_handles_labels(); l2,lb2=ax4b.get_legend_handles_labels()
ax4.legend(l1+l2,lb1+lb2,fontsize=8)

fig.suptitle('So sánh tổng thể — Classical vs Quantum K-Means (Swap Test GPU)',
             fontsize=15,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig('fig1_overall_comparison.png',dpi=150,bbox_inches='tight')
plt.show(); print('fig1 done')

In [ ]:
# ==============================================================================
# CELL 6: Fig 2 — Scatter + Co-occurrence từng k
# ==============================================================================
pc1, pc2 = X[:,0], X[:,1]
fig, axes = plt.subplots(len(valid_ks), 3, figsize=(18, 5*len(valid_ks)))
fig.patch.set_facecolor(BG)

for row, k in enumerate(valid_ks):
    cl_lbl=classical_results[k].labels_; qkm_lbl=quantum_results[k].labels_
    ari_v=adjusted_rand_score(cl_lbl,qkm_lbl)
    for ci,(lbl,ttl,mkr,col) in enumerate([
        (cl_lbl,  f'k={k} | Classical',              'o', COLOR_CL),
        (qkm_lbl, f'k={k} | Quantum (ARI={ari_v:.3f})', '^', COLOR_QK),
    ]):
        ax=axes[row,ci]; ax.set_facecolor('#fff')
        for c in range(k):
            mask=lbl==c
            ax.scatter(pc1[mask],pc2[mask],c=PALETTE[c%len(PALETTE)],
                       s=18,alpha=0.72,marker=mkr)
        ax.set_title(ttl,fontsize=10,fontweight='bold',
                     color=col if ci==1 else '#1a1a2e')
        ax.set_xlabel('PC1',fontsize=9); ax.set_ylabel('PC2',fontsize=9)
        ax.spines[['top','right']].set_visible(False)
    ax=axes[row,2]
    conf=np.zeros((k,k),dtype=int)
    for a,b in zip(cl_lbl,qkm_lbl): conf[a,b]+=1
    im=ax.imshow(conf,cmap='YlOrRd',aspect='auto')
    ax.set_xticks(range(k)); ax.set_yticks(range(k))
    ax.set_xticklabels([f'Q{c}' for c in range(k)],fontsize=8)
    ax.set_yticklabels([f'Cl{c}' for c in range(k)],fontsize=8)
    ax.set_xlabel('Quantum',fontsize=9); ax.set_ylabel('Classical',fontsize=9)
    ax.set_title(f'k={k} | Co-occurrence',fontsize=10,fontweight='bold')
    for i in range(k):
        for j in range(k):
            ax.text(j,i,str(conf[i,j]),ha='center',va='center',fontsize=8,
                    color='white' if conf[i,j]>conf.max()*0.6 else 'black')
    plt.colorbar(im,ax=ax,shrink=0.8)

fig.suptitle('So sánh từng k — Scatter PC1/PC2 & Label Co-occurrence',
             fontsize=15,fontweight='bold',y=1.001)
plt.tight_layout()
plt.savefig('fig2_per_k_comparison.png',dpi=110,bbox_inches='tight')
plt.show(); print('fig2 done')

In [ ]:
# ==============================================================================
# CELL 7: Fig 3 — Convergence
# ==============================================================================
n_cols = 3
n_rows = int(np.ceil(len(valid_ks) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(17, 5*n_rows))
fig.patch.set_facecolor(BG)
axes_flat = np.array(axes).flatten()

for idx, k in enumerate(valid_ks):
    ax=axes_flat[idx]; ax.set_facecolor('#fff')
    ch=classical_results[k].best_run_history_
    qh=quantum_results[k].best_run_history_
    ic=list(range(1,len(ch)+1)); iq=list(range(1,len(qh)+1))
    ax.plot(ic,norm01(ch),'o-', color=COLOR_CL,lw=2,ms=5,label=f'Classical ({len(ch)} iters)')
    ax.plot(iq,norm01(qh),'s--',color=COLOR_QK,lw=2,ms=5,label=f'QKM GPU ({len(qh)} iters)')
    ax.scatter([ic[-1]],[norm01(ch)[-1]],s=80,color=COLOR_CL,zorder=5,edgecolors='white')
    ax.scatter([iq[-1]],[norm01(qh)[-1]],s=80,color=COLOR_QK,zorder=5,edgecolors='white')
    ax.set_title(f'k = {k}',fontsize=12,fontweight='bold')
    ax.set_xlabel('Iteration',fontsize=10); ax.set_ylabel('Inertia (norm)',fontsize=10)
    ax.legend(fontsize=8); ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y',ls='--',alpha=0.4,color='#ccc'); ax.set_ylim(-0.05,1.15)

for j in range(len(valid_ks), len(axes_flat)): axes_flat[j].set_visible(False)

fig.suptitle('Biểu đồ Hội tụ — Classical vs Quantum K-Means',
             fontsize=14,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig('fig3_convergence.png',dpi=150,bbox_inches='tight')
plt.show(); print('fig3 done')

In [ ]:
# ==============================================================================
# CELL 8: Kết luận tự động
# ==============================================================================
print('='*60)
print('  KẾT QUẢ TỔNG HỢP')
print('='*60)

print(f'\nSilhouette trung bình:')
print(f'  Classical : {cl_df["silhouette"].mean():.4f}')
print(f'  Quantum   : {qk_df["silhouette"].mean():.4f}')

print(f'\nDavies-Bouldin trung bình (thấp hơn tốt hơn):')
print(f'  Classical : {cl_df["davies_bouldin"].mean():.4f}')
print(f'  Quantum   : {qk_df["davies_bouldin"].mean():.4f}')

print(f'\nARI trung bình (Classical vs Quantum):')
print(f'  Mean ARI  : {np.mean(ari_arr):.4f}')
print(f'  Min ARI   : {np.min(ari_arr):.4f} (k={valid_ks[np.argmin(ari_arr)]})')
print(f'  Max ARI   : {np.max(ari_arr):.4f} (k={valid_ks[np.argmax(ari_arr)]})')

winner = 'Classical' if cl_df['silhouette'].mean() > qk_df['silhouette'].mean() else 'Quantum'
print(f'\n  Phương pháp tốt hơn tổng thể (Silhouette): {winner}')

print(f'\nBest k theo Silhouette:')
print(f'  Classical : k={cl_df["silhouette"].idxmax()}')
print(f'  Quantum   : k={qk_df["silhouette"].idxmax()}')

In [ ]:
# ==============================================================================
# CELL 9: Lưu kết quả
# ==============================================================================
with open('clustering_results_final.pkl','wb') as f:
    pickle.dump(dict(
        classical_results = classical_results,
        quantum_results   = quantum_results,
        metrics_df        = metrics_df,
        valid_ks          = valid_ks,
    ), f)

metrics_df.to_csv('clustering_metrics.csv', index=False)

print('Da luu:')
for fn in [
    'clustering_results_final.pkl',
    'clustering_metrics.csv',
    'fig1_overall_comparison.png',
    'fig2_per_k_comparison.png',
    'fig3_convergence.png',
]:
    print(f'  {fn}')